# Stage B continual-pretraining baseline gate

This notebook is the required **zero-update evaluation gate** before medical continual pretraining. It restores the promoted Stage A checkpoint, verifies the disjoint Stage B data, and records medical and general validation baselines. It does not train the model.

## Before running

1. In Colab, choose **Runtime → Change runtime type → GPU**.
2. Upload `stage-b-data.tar` to `MyDrive/medical-slm/stage-b-data.tar`.
3. Keep the promoted checkpoint at `MyDrive/medical-slm-runs/stage_a/checkpoints/checkpoint_00007250`.
4. Push the current repository changes to GitHub so the clone includes Stage B support.

The Colab profile is pinned to FP16 with a gradient scaler so an interrupted run remains resumable if Colab changes the assigned GPU between T4 and A100 sessions. Baseline evaluation itself performs no backward pass.

In [1]:
from google.colab import drive
from pathlib import Path
import json
import os
import shutil
import subprocess

drive.mount('/content/drive')

REPOSITORY_URL = 'https://github.com/moshiur00/small-language-model-for-medical-domain.git'
REPOSITORY = Path('/content/medical-slm')
DATA_ARCHIVE = Path('/content/drive/MyDrive/medical-slm/stage-b-data.tar')
DRIVE_PARENT = Path('/content/drive/MyDrive/medical-slm-runs/stage_a/checkpoints/checkpoint_00007250')
LOCAL_PARENT = Path('/content/stage_a_parent/checkpoint_00007250')
LOCAL_REPORT = Path('/content/stage_b/stage_b_baseline.json')
DRIVE_REPORT = Path('/content/drive/MyDrive/medical-slm-runs/stage_b/stage_b_baseline.json')

assert DATA_ARCHIVE.is_file(), f'Missing {DATA_ARCHIVE}'
assert (DRIVE_PARENT / 'checkpoint_manifest.json').is_file(), f'Missing or incomplete promoted Stage A checkpoint: {DRIVE_PARENT}'
print('Inputs found.')

Mounted at /content/drive
Inputs found.


## Download code and install the project

In [2]:
if not (REPOSITORY / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY)], check=True)
else:
    subprocess.run(['git', '-C', str(REPOSITORY), 'pull', '--ff-only'], check=True)

subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', str(REPOSITORY) + '[dev]'], check=True)
os.chdir(REPOSITORY)
print('Repository revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

Repository revision: 12d4b106a423f3ae890fa601f0f8aa8ccae15902


## Restore the data and promoted Stage A parent

In [3]:
subprocess.run(['tar', '-xf', str(DATA_ARCHIVE), '-C', str(REPOSITORY)], check=True)
LOCAL_PARENT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_PARENT, LOCAL_PARENT, dirs_exist_ok=True)

stage_b_shards = sorted(Path('datasets/tokenized/continual_medical_stage_b/train').glob('*.bin'))
assert len(stage_b_shards) == 107, f'Expected 107 Stage B shards, found {len(stage_b_shards)}'
assert Path('datasets/tokenized/evaluation_medical/dataset_manifest.json').is_file()
assert Path('datasets/tokenized/evaluation/dataset_manifest.json').is_file()
assert Path('artifacts/tokenizer/tokenizer.json').is_file()
print('Stage B shards:', len(stage_b_shards))
print('Parent checkpoint restored:', LOCAL_PARENT)

Stage B shards: 107
Parent checkpoint restored: /content/stage_a_parent/checkpoint_00007250


## Verify GPU, precision, tests, and parent initialization

In [6]:
import importlib
import sys
from pathlib import Path

SOURCE_DIRECTORY = Path("/content/medical-slm/src")
assert SOURCE_DIRECTORY.is_dir()

source_path = str(SOURCE_DIRECTORY)
if source_path not in sys.path:
    sys.path.insert(0, source_path)

importlib.invalidate_caches()

import medical_slm

print("medical_slm loaded from:", medical_slm.__file__)

medical_slm loaded from: /content/medical-slm/src/medical_slm/__init__.py


In [7]:
import torch
from medical_slm.training.precision import resolve_precision

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
gpu_name = torch.cuda.get_device_name(0)
precision = resolve_precision('fp16', 'cuda')
print({'gpu': gpu_name, 'precision': precision.name, 'uses_grad_scaler': precision.uses_grad_scaler})

{'gpu': 'Tesla T4', 'precision': 'fp16', 'uses_grad_scaler': True}


In [8]:
subprocess.run([
    'python', '-m', 'pytest',
    'tests/test_stage_b_trainer.py',
    'tests/test_training_config.py',
    'tests/test_training_checkpoint.py',
    '-q',
], check=True)

CompletedProcess(args=['python', '-m', 'pytest', 'tests/test_stage_b_trainer.py', 'tests/test_training_config.py', 'tests/test_training_checkpoint.py', '-q'], returncode=0)

In [9]:
subprocess.run([
    'python', 'scripts/training/train_stage_b.py',
    '--config', 'configs/training_stage_b_colab.yaml',
    '--verify-initialization-only',
], check=True)
print('Parent/model/optimizer initialization gate: PASSED')

Parent/model/optimizer initialization gate: PASSED


## Run the zero-update dual baseline

This evaluates the full medical validation split and the unchanged Stage A general validation split. Depending on the assigned GPU, it can take several minutes.

In [10]:
subprocess.run([
    'python', 'scripts/training/train_stage_b.py',
    '--config', 'configs/training_stage_b_colab.yaml',
    '--baseline-only',
    '--baseline-output', str(LOCAL_REPORT),
], check=True)

CompletedProcess(args=['python', 'scripts/training/train_stage_b.py', '--config', 'configs/training_stage_b_colab.yaml', '--baseline-only', '--baseline-output', '/content/stage_b/stage_b_baseline.json'], returncode=0)

## Verify and preserve the baseline report

In [11]:
import json
import math

report = json.loads(LOCAL_REPORT.read_text())
medical = report['medical_validation']
general = report['general_validation']

assert report['status'] == 'passed'
assert report['optimizer_updates'] == 0
assert report['consumed_tokens'] == 0
assert report['model_parameters'] == 35_463_680
assert report['parent']['checkpoint_name'] == 'checkpoint_00007250'
assert medical['tokens'] == 997_632
assert general['tokens'] == 466_432
for result in (medical, general):
    assert math.isfinite(result['loss']) and math.isfinite(result['perplexity'])

DRIVE_REPORT.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(LOCAL_REPORT, DRIVE_REPORT)
assert DRIVE_REPORT.read_bytes() == LOCAL_REPORT.read_bytes()

print('STAGE B BASELINE GATE: PASSED')
print(f"Medical: loss={medical['loss']:.6f}, perplexity={medical['perplexity']:.3f}, tokens={medical['tokens']}")
print(f"General: loss={general['loss']:.6f}, perplexity={general['perplexity']:.3f}, tokens={general['tokens']}")
print(f"General-loss ceiling during Stage B: {report['general_loss_limit']:.6f}")
print('Saved:', DRIVE_REPORT)

STAGE B BASELINE GATE: PASSED
Medical: loss=3.467505, perplexity=32.057, tokens=997632
General: loss=3.198383, perplexity=24.493, tokens=466432
General-loss ceiling during Stage B: 3.358302
Saved: /content/drive/MyDrive/medical-slm-runs/stage_b/stage_b_baseline.json


## Pass criteria and next action

Proceed to the short Stage B development run only when the final cell prints `STAGE B BASELINE GATE: PASSED`. The two measured losses become immutable run baselines. During continual pretraining, a checkpoint is eligible for promotion only if medical validation improves and general validation loss remains no more than 5% above this baseline.

# One-batch alignment diagnostic

This isolated ten-update run repeatedly trains on one real packed batch. It does not create a resumable checkpoint and cannot contaminate development or full-run state.

In [12]:
import yaml

OVERFIT_OUTPUT = Path('/content/stage_b_overfit')
OVERFIT_CONFIG = Path('/content/training_stage_b_colab_overfit.yaml')
if OVERFIT_OUTPUT.exists():
    shutil.rmtree(OVERFIT_OUTPUT)
overfit_config = yaml.safe_load(Path('configs/training_stage_b_colab.yaml').read_text())
overfit_config.update({
    'output_directory': str(OVERFIT_OUTPUT),
    'checkpoint_backup_directory': None,
    'precision': 'fp16',
    'log_interval': 1,
    'max_updates': 10,
})
OVERFIT_CONFIG.write_text(yaml.safe_dump(overfit_config, sort_keys=False))
subprocess.run([
    'python', 'scripts/training/train_stage_b.py',
    '--config', str(OVERFIT_CONFIG),
    '--overfit-one-batch',
    '--max-updates', '10',
], check=True)

overfit_records = [
    json.loads(line)
    for line in (OVERFIT_OUTPUT / 'metrics.jsonl').read_text().splitlines()
    if line.strip() and json.loads(line)['event'] == 'overfit_one_batch'
]
assert len(overfit_records) == 10
assert all(record['metrics']['optimizer_stepped'] for record in overfit_records)
assert overfit_records[-1]['metrics']['loss'] < overfit_records[0]['metrics']['loss']
print('ONE-BATCH ALIGNMENT GATE: PASSED')
print('loss:', overfit_records[0]['metrics']['loss'], '->', overfit_records[-1]['metrics']['loss'])

ONE-BATCH ALIGNMENT GATE: PASSED
loss: 3.7296557426452637 -> 3.3627419471740723


# Development run: update 0 to 50

Run only after the baseline gate passes. Development checkpoints use `stage_b_dev`, completely separate from the full-run `stage_b` checkpoints.

In [13]:
import yaml

DEV_OUTPUT = Path('/content/stage_b_dev')
DEV_CHECKPOINTS = DEV_OUTPUT / 'checkpoints'
DRIVE_DEV_CHECKPOINTS = Path('/content/drive/MyDrive/medical-slm-runs/stage_b_dev/checkpoints')
DEV_CONFIG = Path('/content/training_stage_b_colab_dev.yaml')

dev_config = yaml.safe_load(Path('configs/training_stage_b_colab.yaml').read_text())
dev_config.update({
    'output_directory': str(DEV_OUTPUT),
    'checkpoint_backup_directory': str(DRIVE_DEV_CHECKPOINTS),
    'precision': 'fp16',
    'max_updates': 100,
    'validation_interval': 50,
    'checkpoint_interval': 50,
})
DEV_CONFIG.write_text(yaml.safe_dump(dev_config, sort_keys=False))

assert not (DEV_CHECKPOINTS / 'latest.json').exists(), 'Local development run exists; use the development resume cell.'
assert not (DRIVE_DEV_CHECKPOINTS / 'latest.json').exists(), 'Drive development run exists; use the development resume cell.'
subprocess.run([
    'python', 'scripts/training/train_stage_b.py',
    '--config', str(DEV_CONFIG),
    '--max-updates', '50',
], check=True)

CompletedProcess(args=['python', 'scripts/training/train_stage_b.py', '--config', '/content/training_stage_b_colab_dev.yaml', '--max-updates', '50'], returncode=0)

In [14]:
from medical_slm.training.checkpoint import verify_checkpoint

dev_pointer = json.loads((DRIVE_DEV_CHECKPOINTS / 'latest.json').read_text())
dev_checkpoint = DRIVE_DEV_CHECKPOINTS / dev_pointer['checkpoint']
verify_checkpoint(dev_checkpoint)
dev_state = json.loads((dev_checkpoint / 'trainer_state.json').read_text())
assert dev_state['update'] == 50
assert dev_state['non_finite_events'] == 0
assert dev_state['skipped_updates'] == 0
print('Development update 50: VERIFIED')
print(dev_state)

Development update 50: VERIFIED
{'batch_cursor': 400, 'best_eligible_medical_loss': 3.440660581556941, 'best_medical_validation_loss': 3.440660581556941, 'best_validation_loss': 3.440660581556941, 'consumed_micro_batches': 400, 'consumed_samples': 6400, 'consumed_tokens': 1638400, 'epoch': 0, 'general_validation_baseline_loss': 3.19838276440017, 'latest_general_validation_loss': 3.226096455514856, 'medical_validation_baseline_loss': 3.4675045480274385, 'non_finite_events': 0, 'skipped_updates': 0, 'update': 50}


In [15]:
print("Medical baseline:", dev_state["medical_validation_baseline_loss"])
print("Best medical:", dev_state["best_medical_validation_loss"])
print("General baseline:", dev_state["general_validation_baseline_loss"])
print("Latest general:", dev_state["latest_general_validation_loss"])
print("Best eligible:", dev_state["best_eligible_medical_loss"])

Medical baseline: 3.4675045480274385
Best medical: 3.440660581556941
General baseline: 3.19838276440017
Latest general: 3.226096455514856
Best eligible: 3.440660581556941


# Development restart/resume: update 50 to 100

After a disconnect, rerun the notebook setup, restore-data, and GPU cells first. This cell restores the verified Drive checkpoint and continues at the next batch cursor.

In [16]:
import json
import subprocess
from pathlib import Path
import yaml
from medical_slm.training.checkpoint import verify_checkpoint

DEV_OUTPUT = Path('/content/stage_b_dev')
DEV_CHECKPOINTS = DEV_OUTPUT / 'checkpoints'
DRIVE_DEV_CHECKPOINTS = Path('/content/drive/MyDrive/medical-slm-runs/stage_b_dev/checkpoints')
DEV_CONFIG = Path('/content/training_stage_b_colab_dev.yaml')
assert (DRIVE_DEV_CHECKPOINTS / 'latest.json').is_file(), 'No development checkpoint exists in Drive.'

dev_config = yaml.safe_load(Path('configs/training_stage_b_colab.yaml').read_text())
dev_config.update({
    'output_directory': str(DEV_OUTPUT),
    'checkpoint_backup_directory': str(DRIVE_DEV_CHECKPOINTS),
    'precision': 'fp16',
    'max_updates': 100,
    'validation_interval': 50,
    'checkpoint_interval': 50,
})
DEV_CONFIG.write_text(yaml.safe_dump(dev_config, sort_keys=False))
DEV_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
subprocess.run(['rsync', '-a', f'{DRIVE_DEV_CHECKPOINTS}/', f'{DEV_CHECKPOINTS}/'], check=True)

subprocess.run([
    'python', 'scripts/training/train_stage_b.py',
    '--config', str(DEV_CONFIG),
    '--resume', 'latest',
    '--max-updates', '100',
], check=True)

dev_pointer = json.loads((DRIVE_DEV_CHECKPOINTS / 'latest.json').read_text())
dev_checkpoint = DRIVE_DEV_CHECKPOINTS / dev_pointer['checkpoint']
verify_checkpoint(dev_checkpoint)
dev_state = json.loads((dev_checkpoint / 'trainer_state.json').read_text())
assert dev_state['update'] == 100
assert dev_state['non_finite_events'] == 0
assert dev_state['skipped_updates'] == 0
print('Development 50 to 100 resume: VERIFIED')

Development 50 to 100 resume: VERIFIED


In [17]:
print("Update:", dev_state["update"])
print("Batch cursor:", dev_state["batch_cursor"])
print("Consumed tokens:", dev_state["consumed_tokens"])
print("Best medical:", dev_state["best_medical_validation_loss"])
print("Latest general:", dev_state["latest_general_validation_loss"])
print("Best eligible:", dev_state["best_eligible_medical_loss"])
print("Skipped:", dev_state["skipped_updates"])
print("Non-finite:", dev_state["non_finite_events"])

Update: 100
Batch cursor: 800
Consumed tokens: 3276800
Best medical: 3.4278715484601623
Latest general: 3.258460616783119
Best eligible: 3.4278715484601623
Skipped: 0
Non-finite: 0


In [18]:
records = [
    json.loads(line)
    for line in Path("/content/stage_b_dev/metrics.jsonl").read_text().splitlines()
    if line.strip()
]

for event in ("medical_validation", "general_retention_validation"):
    result = [r for r in records if r["event"] == event][-1]
    print(event, "update:", result["update"], result["metrics"])

medical_validation update: 100 {'duration_seconds': 9.466634482000018, 'general_baseline_loss': 3.19838276440017, 'general_loss_limit': 3.3583019026201786, 'is_best_eligible': True, 'is_best_medical': True, 'loss': 3.4278715484601623, 'medical_baseline_loss': 3.4675045480274385, 'perplexity': 30.810993202595636, 'promotion_eligible': True, 'samples': 3897, 'tokens': 997632}
general_retention_validation update: 100 {'duration_seconds': 4.671547043000146, 'general_baseline_loss': 3.19838276440017, 'general_loss_limit': 3.3583019026201786, 'loss': 3.258460616783119, 'medical_baseline_loss': 3.4675045480274385, 'perplexity': 26.00946777120519, 'promotion_eligible': True, 'relative_loss_change': 0.01878382195265993, 'samples': 1822, 'tokens': 466432}


# Fresh full Stage B training — standalone

Run this cell only after the baseline, update-50, and 50→100 resume gates pass. It performs one full epoch (6,840 optimizer updates). It deliberately refuses to start if a full-run checkpoint already exists; use the resume cell in that case.

In [19]:
from google.colab import drive
from pathlib import Path
import json
import os
import shutil
import subprocess
import torch

drive.mount('/content/drive')
REPOSITORY_URL = 'https://github.com/moshiur00/small-language-model-for-medical-domain.git'
REPOSITORY = Path('/content/medical-slm')
DATA_ARCHIVE = Path('/content/drive/MyDrive/medical-slm/stage-b-data.tar')
DRIVE_PARENT = Path('/content/drive/MyDrive/medical-slm-runs/stage_a/checkpoints/checkpoint_00007250')
LOCAL_PARENT = Path('/content/stage_a_parent/checkpoint_00007250')
LOCAL_FULL_CHECKPOINTS = Path('/content/stage_b/checkpoints')
DRIVE_FULL_CHECKPOINTS = Path('/content/drive/MyDrive/medical-slm-runs/stage_b/checkpoints')
DRIVE_BASELINE = Path('/content/drive/MyDrive/medical-slm-runs/stage_b/stage_b_baseline.json')
FULL_CONFIG = Path('configs/training_stage_b_colab.yaml')

assert DATA_ARCHIVE.is_file(), f'Missing {DATA_ARCHIVE}'
assert (DRIVE_PARENT / 'checkpoint_manifest.json').is_file(), f'Missing parent {DRIVE_PARENT}'
assert DRIVE_BASELINE.is_file(), 'Run and preserve the Stage B baseline gate first.'
baseline_gate = json.loads(DRIVE_BASELINE.read_text())
assert baseline_gate['status'] == 'passed' and baseline_gate['optimizer_updates'] == 0
assert not (LOCAL_FULL_CHECKPOINTS / 'latest.json').exists(), 'Local full run exists; use the resume cell.'
assert not (DRIVE_FULL_CHECKPOINTS / 'latest.json').exists(), 'Drive full run exists; use the resume cell.'
assert torch.cuda.is_available(), 'Select a GPU runtime before training.'

if not (REPOSITORY / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY)], check=True)
else:
    subprocess.run(['git', '-C', str(REPOSITORY), 'pull', '--ff-only'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', str(REPOSITORY) + '[dev]'], check=True)
os.chdir(REPOSITORY)
subprocess.run(['tar', '-xf', str(DATA_ARCHIVE), '-C', str(REPOSITORY)], check=True)
LOCAL_PARENT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_PARENT, LOCAL_PARENT, dirs_exist_ok=True)

print('GPU:', torch.cuda.get_device_name(0))
print('Configured precision: fp16')
subprocess.run(['python', 'scripts/training/train_stage_b.py', '--config', str(FULL_CONFIG), '--verify-initialization-only'], check=True)
subprocess.run(['python', 'scripts/training/train_stage_b.py', '--config', str(FULL_CONFIG)], check=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
Configured precision: fp16


CompletedProcess(args=['python', 'scripts/training/train_stage_b.py', '--config', 'configs/training_stage_b_colab.yaml'], returncode=0)

# Resume interrupted full Stage B training — standalone

Use this cell in a new GPU runtime after interruption. It downloads the current code, restores all required data and the Stage A parent, verifies the latest Stage B checkpoint, and resumes with the same FP16 policy.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import os
import shutil
import subprocess
import torch
import yaml

drive.mount('/content/drive')
REPOSITORY_URL = 'https://github.com/moshiur00/small-language-model-for-medical-domain.git'
REPOSITORY = Path('/content/medical-slm')
DATA_ARCHIVE = Path('/content/drive/MyDrive/medical-slm/stage-b-data.tar')
DRIVE_PARENT = Path('/content/drive/MyDrive/medical-slm-runs/stage_a/checkpoints/checkpoint_00007250')
LOCAL_PARENT = Path('/content/stage_a_parent/checkpoint_00007250')
LOCAL_FULL_CHECKPOINTS = Path('/content/stage_b/checkpoints')
DRIVE_FULL_CHECKPOINTS = Path('/content/drive/MyDrive/medical-slm-runs/stage_b/checkpoints')
RESUME_CONFIG = Path('/content/training_stage_b_colab_resume.yaml')

assert DATA_ARCHIVE.is_file(), f'Missing {DATA_ARCHIVE}'
assert (DRIVE_PARENT / 'checkpoint_manifest.json').is_file(), f'Missing parent {DRIVE_PARENT}'
assert (DRIVE_FULL_CHECKPOINTS / 'latest.json').is_file(), 'No full Stage B checkpoint exists in Drive.'
assert torch.cuda.is_available(), 'Select a GPU runtime before resuming.'

if not (REPOSITORY / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY)], check=True)
else:
    subprocess.run(['git', '-C', str(REPOSITORY), 'pull', '--ff-only'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', str(REPOSITORY) + '[dev]'], check=True)
os.chdir(REPOSITORY)
subprocess.run(['tar', '-xf', str(DATA_ARCHIVE), '-C', str(REPOSITORY)], check=True)
LOCAL_PARENT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_PARENT, LOCAL_PARENT, dirs_exist_ok=True)
LOCAL_FULL_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
subprocess.run(['rsync', '-a', f'{DRIVE_FULL_CHECKPOINTS}/', f'{LOCAL_FULL_CHECKPOINTS}/'], check=True)

from medical_slm.training.checkpoint import verify_checkpoint
latest_pointer = json.loads((LOCAL_FULL_CHECKPOINTS / 'latest.json').read_text())
latest_name = latest_pointer['checkpoint']
latest_checkpoint = LOCAL_FULL_CHECKPOINTS / latest_name
verify_checkpoint(latest_checkpoint)
saved_config = json.loads((latest_checkpoint / 'config.json').read_text())
assert saved_config['training']['precision'] == 'fp16', 'Checkpoint precision is not FP16.'
RESUME_CONFIG.write_text(yaml.safe_dump(saved_config['training'], sort_keys=False))
state = json.loads((latest_checkpoint / 'trainer_state.json').read_text())
print('GPU:', torch.cuda.get_device_name(0))
print('Restored:', latest_name, 'update:', state['update'], 'tokens:', state['consumed_tokens'])

subprocess.run([
    'python', 'scripts/training/train_stage_b.py',
    '--config', str(RESUME_CONFIG),
    '--resume', 'latest',
], check=True)

## Full-run completion check

When training finishes, Drive must contain `final_stage_b.json`. Do not evaluate the sealed medical test split until validation-based checkpoint selection is complete.

In [20]:
import json
from pathlib import Path
from medical_slm.training.checkpoint import verify_checkpoint

DRIVE_FULL_CHECKPOINTS = Path('/content/drive/MyDrive/medical-slm-runs/stage_b/checkpoints')
final_pointer_path = DRIVE_FULL_CHECKPOINTS / 'final_stage_b.json'
assert final_pointer_path.is_file(), 'Full Stage B training is not complete yet.'
final_pointer = json.loads(final_pointer_path.read_text())
final_checkpoint = DRIVE_FULL_CHECKPOINTS / final_pointer['checkpoint']
verify_checkpoint(final_checkpoint)
final_state = json.loads((final_checkpoint / 'trainer_state.json').read_text())
assert final_state['update'] == 6840
assert final_state['epoch'] == 1
assert final_state['non_finite_events'] == 0
assert final_state['skipped_updates'] == 0
print('FULL STAGE B STATE: VERIFIED')
print(final_state)

FULL STAGE B STATE: VERIFIED
{'batch_cursor': 0, 'best_eligible_medical_loss': 3.374684920068472, 'best_medical_validation_loss': 3.0091672630716295, 'best_validation_loss': 3.0091672630716295, 'consumed_micro_batches': 54717, 'consumed_samples': 875470, 'consumed_tokens': 224120320, 'epoch': 1, 'general_validation_baseline_loss': 3.19838276440017, 'latest_general_validation_loss': 3.7491228888529475, 'medical_validation_baseline_loss': 3.4675045480274385, 'non_finite_events': 0, 'skipped_updates': 0, 'update': 6840}


In [21]:
import json
from pathlib import Path

root = Path(
    "/content/drive/MyDrive/medical-slm-runs/stage_b/checkpoints"
)

pointer = json.loads((root / "best_eligible.json").read_text())
checkpoint = root / pointer["checkpoint"]
state = json.loads((checkpoint / "trainer_state.json").read_text())

print("Best eligible checkpoint:", checkpoint.name)
print("Update:", state["update"])
print("Medical loss:", state["best_eligible_medical_loss"])
print("General loss:", state["latest_general_validation_loss"])
print(
    "General ceiling:",
    state["general_validation_baseline_loss"] * 1.05,
)

assert state["latest_general_validation_loss"] <= (
    state["general_validation_baseline_loss"] * 1.05
)

Best eligible checkpoint: checkpoint_00000250
Update: 250
Medical loss: 3.374684920068472
General loss: 3.3300274743562572
General ceiling: 3.3583019026201786


In [22]:
from medical_slm.training.checkpoint import verify_checkpoint

manifest = verify_checkpoint(checkpoint)

print("Checkpoint integrity: VERIFIED")
print("Artifacts:", len(manifest["artifacts"]))
print("Lineage:", manifest["lineage"])

Checkpoint integrity: VERIFIED
Artifacts: 9
Lineage: {'data': {'general_validation_manifest_sha256': '440cde59138604720d0afbceb14777515c8591b2ff71000dc158f427994f8ffd', 'medical_validation_manifest_sha256': '6537ac466894df9175009017198e7c9996e55705a9a9388130716f4e261a706d', 'train_manifest_sha256': '1d8b828604c8e6ff9cc6ea4d6cffe604f78367dd87e8c171d73dd5dcfd4170d1'}, 'parent': {'checkpoint_manifest_sha256': 'a7e132692f31b505b7d7db9fa7e6d773d5c006904fa0d774edb1ba6c7f0408a4', 'checkpoint_name': 'checkpoint_00007250', 'model_sha256': '2443cb5875c11e9c0c027ead53d4f9adab099e0cd4b19fd47fe08181b0640423', 'tokenizer_sha256': '6c569241e2d166cfba709d8d260cdcbdd6b0907ce45dfa644e0426f1aecb078e'}, 'stage': 'continual_medical_stage_b'}


# Validation

In [23]:
import json
import math
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import torch
import yaml
from torch.utils.data import DataLoader

from medical_slm.data.tokenization import (
    PackedTokenDataset,
    calculate_sha256,
)
from medical_slm.model import DecoderConfig, DecoderModel
from medical_slm.training.checkpoint import load_model_weights
from medical_slm.training.evaluation import evaluate_shifted_packed
from medical_slm.training.precision import resolve_precision

# ---------- Candidate checkpoint ----------

root = Path(
    "/content/drive/MyDrive/medical-slm-runs/stage_b/checkpoints"
)

pointer = json.loads((root / "best_eligible.json").read_text())
checkpoint = root / pointer["checkpoint"]

# ---------- Model-only loading ----------

device = torch.device("cuda")
precision = resolve_precision("fp16", device)

model_values = yaml.safe_load(
    Path("configs/model_stage_a.yaml").read_text()
)
model_config = DecoderConfig.from_mapping(model_values)

model = DecoderModel(model_config).to(device)

tokenizer_hash = calculate_sha256(
    Path("artifacts/tokenizer/tokenizer.json")
)

loaded_checkpoint = load_model_weights(
    checkpoint_directory=checkpoint,
    model=model,
    expected_model_config=model_config.to_dict(),
    expected_tokenizer_sha256=tokenizer_hash,
    map_location=device,
)

assert model.parameter_count() == 35_463_680

# ---------- Deterministic validation loaders ----------

def validation_loader(path: str) -> DataLoader:
    dataset = PackedTokenDataset(path)
    return DataLoader(
        dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

medical_result = evaluate_shifted_packed(
    model=model,
    batches=validation_loader(
        "datasets/tokenized/evaluation_medical/validation"
    ),
    device=device,
    precision=precision,
)

general_result = evaluate_shifted_packed(
    model=model,
    batches=validation_loader(
        "datasets/tokenized/evaluation/validation"
    ),
    device=device,
    precision=precision,
)

# ---------- Promotion gate ----------

medical_baseline = 3.4675045480274385
general_baseline = 3.19838276440017
general_ceiling = general_baseline * 1.05

assert medical_result.tokens == 997_632
assert general_result.tokens == 466_432
assert math.isfinite(medical_result.loss)
assert math.isfinite(general_result.loss)

medical_passed = medical_result.loss < medical_baseline
retention_passed = general_result.loss <= general_ceiling
validation_passed = medical_passed and retention_passed

assert validation_passed

print("CANDIDATE VALIDATION: PASSED")
print("Checkpoint:", checkpoint.name)
print(
    f"Medical: loss={medical_result.loss:.6f}, "
    f"perplexity={medical_result.perplexity:.3f}"
)
print(
    f"General: loss={general_result.loss:.6f}, "
    f"perplexity={general_result.perplexity:.3f}"
)
print("General ceiling:", general_ceiling)

# ---------- Preserve independent report ----------

report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "checkpoint": checkpoint.name,
    "checkpoint_identity": loaded_checkpoint,
    "precision": precision.name,
    "medical_validation": asdict(medical_result),
    "general_validation": asdict(general_result),
    "medical_baseline_loss": medical_baseline,
    "general_baseline_loss": general_baseline,
    "general_loss_ceiling": general_ceiling,
    "medical_passed": medical_passed,
    "retention_passed": retention_passed,
    "validation_passed": validation_passed,
}

report_path = Path(
    "/content/drive/MyDrive/medical-slm-runs/"
    "stage_b/stage_b_candidate_validation.json"
)
report_path.write_text(
    json.dumps(report, indent=2, sort_keys=True) + "\n"
)

print("Saved:", report_path)

CANDIDATE VALIDATION: PASSED
Checkpoint: checkpoint_00000250
Medical: loss=3.374685, perplexity=29.215
General: loss=3.330027, perplexity=27.939
General ceiling: 3.3583019026201786
Saved: /content/drive/MyDrive/medical-slm-runs/stage_b/stage_b_candidate_validation.json


In [24]:
import json
import math
import os
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

from medical_slm.training.evaluation import evaluate_shifted_packed

# ---------- One-time sealed medical-test evaluation ----------

medical_test_result = evaluate_shifted_packed(
    model=model,
    batches=validation_loader(
        "datasets/tokenized/evaluation_medical/test"
    ),
    device=device,
    precision=precision,
)

assert medical_test_result.samples == 3_926
assert medical_test_result.tokens == 1_005_056
assert math.isfinite(medical_test_result.loss)
assert math.isfinite(medical_test_result.perplexity)

print("SEALED MEDICAL TEST: EVALUATED")
print("Checkpoint:", checkpoint.name)
print("Test loss:", medical_test_result.loss)
print("Test perplexity:", medical_test_result.perplexity)
print("Test samples:", medical_test_result.samples)
print("Test tokens:", medical_test_result.tokens)

# ---------- Final Stage B reports ----------

candidate_report_path = Path(
    "/content/drive/MyDrive/medical-slm-runs/"
    "stage_b/stage_b_candidate_validation.json"
)
candidate_report = json.loads(candidate_report_path.read_text())

evaluation_report = {
    "stage": "continual_medical_stage_b",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "selected_checkpoint": checkpoint.name,
    "selection_rule": (
        "Lowest medical-validation loss among checkpoints whose "
        "general-validation loss remained within 5% of the Stage A baseline."
    ),
    "checkpoint_identity": loaded_checkpoint,
    "candidate_validation": {
        "medical": candidate_report["medical_validation"],
        "general": candidate_report["general_validation"],
        "medical_baseline_loss": candidate_report[
            "medical_baseline_loss"
        ],
        "general_baseline_loss": candidate_report[
            "general_baseline_loss"
        ],
        "general_loss_ceiling": candidate_report[
            "general_loss_ceiling"
        ],
        "validation_passed": candidate_report[
            "validation_passed"
        ],
    },
    "medical_test": asdict(medical_test_result),
    "test_evaluated_once": True,
    "limitations": [
        "Loss and perplexity do not establish medical factuality.",
        "The model has not been evaluated for clinical safety.",
        "The checkpoint is a base model, not a medical assistant.",
    ],
}

promotion_pointer = {
    "stage": "continual_medical_stage_b",
    "checkpoint": checkpoint.name,
    "evaluation_report": "stage_b_evaluation.json",
    "validation_selected": True,
    "medical_test_used_for_selection": False,
}

drive_stage_b = Path(
    "/content/drive/MyDrive/medical-slm-runs/stage_b"
)
drive_stage_b.mkdir(parents=True, exist_ok=True)

def atomic_json(path: Path, payload: dict) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True) + "\n"
    )
    os.replace(temporary, path)

atomic_json(
    drive_stage_b / "stage_b_evaluation.json",
    evaluation_report,
)
atomic_json(
    drive_stage_b / "promoted_stage_b.json",
    promotion_pointer,
)

print("Saved:", drive_stage_b / "stage_b_evaluation.json")
print("Saved:", drive_stage_b / "promoted_stage_b.json")
print("STAGE B PROMOTION ARTIFACTS: WRITTEN")

SEALED MEDICAL TEST: EVALUATED
Checkpoint: checkpoint_00000250
Test loss: 3.3954293786604297
Test perplexity: 29.827457999706436
Test samples: 3926
Test tokens: 1005056
Saved: /content/drive/MyDrive/medical-slm-runs/stage_b/stage_b_evaluation.json
Saved: /content/drive/MyDrive/medical-slm-runs/stage_b/promoted_stage_b.json
STAGE B PROMOTION ARTIFACTS: WRITTEN


In [25]:
import json
from pathlib import Path

records = [
    json.loads(line)
    for line in Path("/content/stage_b/metrics.jsonl").read_text().splitlines()
    if line.strip()
]

medical = {
    record["update"]: record["metrics"]
    for record in records
    if record["event"] == "medical_validation"
}

general = {
    record["update"]: record["metrics"]
    for record in records
    if record["event"] == "general_retention_validation"
}

medical_baseline = 3.4675045480274385
general_baseline = 3.19838276440017

print(
    "update | medical | med improvement | general | general degradation"
)

for update in sorted(medical.keys() & general.keys()):
    medical_loss = medical[update]["loss"]
    general_loss = general[update]["loss"]

    medical_improvement = (
        1 - medical_loss / medical_baseline
    ) * 100

    general_degradation = (
        general_loss / general_baseline - 1
    ) * 100

    print(
        f"{update:6d} | "
        f"{medical_loss:.6f} | "
        f"{medical_improvement:6.2f}% | "
        f"{general_loss:.6f} | "
        f"{general_degradation:6.2f}%"
    )

update | medical | med improvement | general | general degradation
     0 | 3.467505 |   0.00% | 3.198383 |   0.00%
   250 | 3.374685 |   2.68% | 3.330027 |   4.12%
   500 | 3.321770 |   4.20% | 3.424986 |   7.08%
   750 | 3.283375 |   5.31% | 3.507547 |   9.67%
  1000 | 3.253855 |   6.16% | 3.563037 |  11.40%
  1250 | 3.224847 |   7.00% | 3.601495 |  12.60%
  1500 | 3.202041 |   7.66% | 3.617688 |  13.11%
  1750 | 3.180341 |   8.28% | 3.650974 |  14.15%
  2000 | 3.163304 |   8.77% | 3.689261 |  15.35%
  2250 | 3.146324 |   9.26% | 3.692723 |  15.46%
  2500 | 3.130781 |   9.71% | 3.705996 |  15.87%
  2750 | 3.116613 |  10.12% | 3.715715 |  16.17%
  3000 | 3.104126 |  10.48% | 3.717768 |  16.24%
  3250 | 3.093059 |  10.80% | 3.722091 |  16.37%
  3500 | 3.081359 |  11.14% | 3.715421 |  16.17%
  3750 | 3.072389 |  11.39% | 3.732413 |  16.70%
  4000 | 3.061555 |  11.71% | 3.739300 |  16.91%
  4250 | 3.054175 |  11.92% | 3.747427 |  17.17%
  4500 | 3.046145 |  12.15% | 3.747891 |  17.18%
  

In [26]:
from pathlib import Path
import shutil

source = Path("/content/stage_b/metrics.jsonl")
destination = Path(
    "/content/drive/MyDrive/medical-slm-runs/"
    "stage_b/v1_source/metrics.jsonl"
)

assert source.is_file() and source.stat().st_size > 0
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source, destination)
assert source.read_bytes() == destination.read_bytes()

print("Metrics preserved:", destination)
print("Bytes:", destination.stat().st_size)

Metrics preserved: /content/drive/MyDrive/medical-slm-runs/stage_b/v1_source/metrics.jsonl
Bytes: 250223


In [27]:
import os
import shutil
import subprocess
from pathlib import Path

repository = Path("/content/medical-slm")
os.chdir(repository)

subprocess.run(
    ["git", "pull", "--ff-only"],
    check=True,
)

bundle = Path("/content/stage_b_v1")
archive = Path("/content/stage_b_v1.tar")

assert not bundle.exists()
assert not archive.exists()

subprocess.run(
    [
        "python",
        "scripts/artifacts/export_stage_b_v1.py",
        "--checkpoint-root",
        "/content/drive/MyDrive/medical-slm-runs/stage_b/checkpoints",
        "--run-output",
        "/content/drive/MyDrive/medical-slm-runs/stage_b/v1_source",
        "--report-root",
        "/content/drive/MyDrive/medical-slm-runs/stage_b",
        "--destination",
        str(bundle),
        "--archive",
        str(archive),
    ],
    check=True,
)

drive_preservation = Path(
    "/content/drive/MyDrive/medical-slm-runs/stage_b/v1_preservation"
)
drive_preservation.mkdir(parents=True, exist_ok=True)

for source in (archive, Path(str(archive) + ".sha256")):
    shutil.copy2(source, drive_preservation / source.name)

print("Preservation archive copied to:", drive_preservation)

Preservation archive copied to: /content/drive/MyDrive/medical-slm-runs/stage_b/v1_preservation
